In [1]:

import numpy as np
import stim
import torch
import pymatching
from collections import OrderedDict
from torch import nn, optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, Subset
import simulated_annealing

In [35]:
def generate_dataset_and_dem(noise_model, code_distance, noise_param):
    if noise_model == 'DP':
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z", # Protect a qubit stored in the z basis to detect a bit flip (logical x error)
            rounds=9,
            distance=code_distance,
            before_round_data_depolarization=noise_param
        )
    elif noise_model == 'FT':
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z", # Protect a qubit stored in the z basis to detect a bit flip (logical x error)
            rounds=9,
            distance=code_distance,
            after_clifford_depolarization=noise_param, # Circuit-level (FT) noise
            after_reset_flip_probability=noise_param, # Circuit-level (FT) noise
            before_measure_flip_probability=noise_param, # Circuit-level (FT) noise
            before_round_data_depolarization=noise_param
        )

    sampler = circuit.compile_detector_sampler(seed=42) #Set a seed so that things are predictable

    # Measuring the stabilizers x- and z-ancillas. 
    # Generating the samples to train the MLP decoder

    n_shots = 1_000_000
    syndromes, obs = sampler.sample(shots=n_shots, separate_observables=True)

    # Converting to numerical numbers for computations 
    labels = obs.astype(np.float32)
    features = syndromes.astype(np.float32)
  
    # From numpy to torch
    labels = torch.from_numpy(labels)
    features = torch.from_numpy(features) 

    dem = circuit.detector_error_model(decompose_errors=True)

    return features, labels, dem

In [3]:
# Depolarizing d = 3, noise_param = 0.005

features, obs, dem = generate_dataset_and_dem('DP', 3, 0.005)

# Dataset from syndromes measurement and logical observables

class quantum_dataset(Dataset):
    def __init__(self):
        self.features = features
        self.obs = obs
    def __len__(self):
        return len(self.features)
    def __getitem__(self, id):
        return self.features[id], self.obs[id]

dataset = quantum_dataset()

BATCH_SIZE = 128
indices = np.arange(len(dataset))
train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=dataset.obs.numpy()
)

train_dataset = Subset(dataset, train_indices)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_dataset = Subset(dataset, test_indices)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [39]:
# Depolarizing d = 3, noise_param = 0.005

features_d3_005, obs_d3_005, dem_d3_005 = generate_dataset_and_dem('DP', 3, 0.005)

# Neural Network Model
input_size = 72 # N_rounds = 9, N_ancillas = 8 (for distance=3)
output_size = 1
hidden_sizes = (512, 256, 128, 64)
dropout_p = [.3, 0.0, .1, .1]
model_d3_005_DP = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], output_size)),
            ])
        )
state_dict = torch.load('depolarizing/model-d3-005.pth', map_location=torch.device('cpu'))
model_d3_005_DP.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\4189403595.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('depolarizing/model-d3-005.pth', 

<All keys matched successfully>

In [31]:
# Depolarizing d = 3, noise_param = 0.01

features_d3_01, obs_d3_01, dem_d3_01 = generate_dataset_and_dem('DP', 3, 0.01)

# Neural Network Model
input_size = 72 # N_rounds = 9, N_ancillas = 8 (for distance=3)
output_size = 1
hidden_sizes = (512, 256, 128, 64)
dropout_p = [.3, 0.0, .1, .1]
model_d3_01_DP = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], output_size)),
            ])
        )
state_dict = torch.load('depolarizing/model-d3-01.pth', map_location=torch.device('cpu'))
model_d3_01_DP.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\1838185470.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('depolarizing/model-d3-01.pth', m

<All keys matched successfully>

In [32]:
# Depolarizing d = 5, noise_param = 0.005

features_d5_005, obs_d5_005, dem_d5_005 = generate_dataset_and_dem('DP', 5, 0.005)

# Neural Network Model
input_size = 216 # N_rounds = 9, N_ancillas = 24 (for distance=5)
output_size = 1
hidden_sizes = (512, 256, 128, 64)
dropout_p = [.3, 0.0, .1, .1]
model_d5_005_DP = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], output_size)),
            ])
        )
state_dict = torch.load('depolarizing/model-d5-005.pth', map_location=torch.device('cpu'))
model_d5_005_DP.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\522707140.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('depolarizing/model-d5-005.pth', m

<All keys matched successfully>

In [33]:
# Depolarizing d = 5, noise_param = 0.01

features_d5_01, obs_d5_01, dem_d5_01 = generate_dataset_and_dem('DP', 5, 0.01)

# Neural Network Model
input_size = 216 # N_rounds = 9, N_ancillas = 24 (for distance=5)
output_size = 1
hidden_sizes = (512, 256, 128, 64)
dropout_p = [.3, 0.0, .1, .1]
model_d5_01_DP = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], output_size)),
            ])
        )
state_dict = torch.load('depolarizing/model-d5-01.pth', map_location=torch.device('cpu'))
model_d5_01_DP.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\808998950.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('depolarizing/model-d5-01.pth', ma

<All keys matched successfully>

In [43]:
# FT d = 3, noise_param = 0.005

features_d3_005_FT, obs_d3_005_FT, dem_d3_005_FT = generate_dataset_and_dem('FT', 3, 0.005)

# Neural Network Model
input_size = 72 # N_rounds = 9, N_ancillas = 8 (for distance=3)
output_size = 1
hidden_sizes = (512, 256, 128, 64, 32)
dropout_p = [.2, .2, .2, .1, .3]
model_d3_005_FT = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], hidden_sizes[4])),
            ('relu5', nn.ReLU()),
            ('drp5', nn.Dropout(dropout_p[4])),
            ('fc6', nn.Linear(hidden_sizes[4], output_size)),
            ])
        )
state_dict = torch.load('FT/model-d3-005.pth', map_location=torch.device('cpu'))
model_d3_005_FT.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\2186746973.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('FT/model-d3-005.pth', map_locati

<All keys matched successfully>

In [44]:
# FT d = 3, noise_param = 0.01

features_d3_01_FT, obs_d3_01_FT, dem_d3_01_FT = generate_dataset_and_dem('FT', 3, 0.01)

# Neural Network Model
input_size = 72 # N_rounds = 9, N_ancillas = 8 (for distance=3)
output_size = 1
hidden_sizes = (512, 256, 128, 64, 32)
dropout_p = [.2, .2, .2, .1, .3]
model_d3_01_FT = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], hidden_sizes[3])),
            ('relu4', nn.ReLU()),
            ('drp4', nn.Dropout(dropout_p[3])),
            ('fc5', nn.Linear(hidden_sizes[3], hidden_sizes[4])),
            ('relu5', nn.ReLU()),
            ('drp5', nn.Dropout(dropout_p[4])),
            ('fc6', nn.Linear(hidden_sizes[4], output_size)),
            ])
        )
state_dict = torch.load('FT/model-d3-01.pth', map_location=torch.device('cpu'))
model_d3_01_FT.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\4031008974.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('FT/model-d3-01.pth', map_locatio

<All keys matched successfully>

In [51]:
# FT d = 5, noise_param = 0.005

features_d5_005_FT, obs_d5_005_FT, dem_d5_005_FT = generate_dataset_and_dem('FT', 5, 0.005)

# Neural Network Model
input_size = 216 # N_rounds = 9, N_ancillas = 24 (for distance=3)
output_size = 1
hidden_sizes = (256, 128, 64)
dropout_p = [.3, .0, .0]
model_d5_005_FT = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], output_size)),
            ])
        )
state_dict = torch.load('FT/model-d5-005.pth', map_location=torch.device('cpu'))
model_d5_005_FT.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\2055105281.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('FT/model-d5-005.pth', map_locati

<All keys matched successfully>

In [52]:
# FT d = 5, noise_param = 0.01

features_d5_01_FT, obs_d5_01_FT, dem_d5_01_FT = generate_dataset_and_dem('FT', 5, 0.01)

# Neural Network Model
input_size = 216 # N_rounds = 9, N_ancillas = 24 (for distance=3)
output_size = 1
hidden_sizes = (256, 128, 64)
dropout_p = [.3, .0, .0]
model_d5_01_FT = nn.Sequential(OrderedDict([
            ('fc1', nn.Linear(input_size, hidden_sizes[0])),
            ('relu1', nn.ReLU()),
            ('drp1', nn.Dropout(dropout_p[0])),
            ('fc2', nn.Linear(hidden_sizes[0], hidden_sizes[1])),
            ('relu2', nn.ReLU()),
            ('drp2', nn.Dropout(dropout_p[1])),
            ('fc3', nn.Linear(hidden_sizes[1], hidden_sizes[2])),
            ('relu3', nn.ReLU()),
            ('drp3', nn.Dropout(dropout_p[2])),
            ('fc4', nn.Linear(hidden_sizes[2], output_size)),
            ])
        )
state_dict = torch.load('FT/model-d5-01.pth', map_location=torch.device('cpu'))
model_d5_01_FT.load_state_dict(state_dict)

C:\Users\Cesaire\AppData\Local\Temp\ipykernel_46136\2202704557.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('FT/model-d5-01.pth', map_locatio

<All keys matched successfully>

In [15]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d3_005_DP, dem_d3_005, criterion, 1)

  restart 1/5 | best BCE loss so far: 0.0198
  restart 2/5 | best BCE loss so far: 0.0198
  restart 3/5 | best BCE loss so far: 0.0198
  restart 4/5 | best BCE loss so far: 0.0198
  restart 5/5 | best BCE loss so far: 0.0198


In [ ]:
print(best_label)
model_d3_005_DP(best_S)

tensor([1.])


tensor([5.3457], grad_fn=<ViewBackward0>)

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d3_01_DP, dem_d3_01, criterion, 2)

  restart 1/5 | best BCE loss so far: 6.6387
  restart 2/5 | best BCE loss so far: 6.6387
  restart 3/5 | best BCE loss so far: 7.8280
  restart 4/5 | best BCE loss so far: 7.8280
  restart 5/5 | best BCE loss so far: 7.8280


In [ ]:
print(best_label)
model_d3_01_DP(best_S)

tensor([1.])


tensor([-7.1783], grad_fn=<ViewBackward0>)

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d5_005_DP, dem_d5_005, criterion, 2)

  restart 1/5 | best BCE loss so far: 0.4554
  restart 2/5 | best BCE loss so far: 0.4554
  restart 3/5 | best BCE loss so far: 0.4554
  restart 4/5 | best BCE loss so far: 0.4554
  restart 5/5 | best BCE loss so far: 0.6197


In [ ]:
print(best_label)
model_d5_005_DP(best_S)

tensor([0.])


tensor([0.1547], grad_fn=<ViewBackward0>)

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d5_01_DP, dem_d5_01, criterion, 2)

  restart 1/5 | best BCE loss so far: 0.3981
  restart 2/5 | best BCE loss so far: 3.2750
  restart 3/5 | best BCE loss so far: 3.2750
  restart 4/5 | best BCE loss so far: 3.2750
  restart 5/5 | best BCE loss so far: 3.2750


In [40]:
print(best_label)
model_d5_01_DP(best_S)

tensor([1.])


tensor([-4.6789], grad_fn=<ViewBackward0>)

In [41]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d3_005_FT, dem_d3_005_FT, criterion, 2)

  restart 1/5 | best BCE loss so far: 10.0435
  restart 2/5 | best BCE loss so far: 10.0435
  restart 3/5 | best BCE loss so far: 11.9626
  restart 4/5 | best BCE loss so far: 11.9626
  restart 5/5 | best BCE loss so far: 11.9626


In [45]:
print(best_label)
model_d3_005_FT(best_S)

tensor([0.])


tensor([8.3440], grad_fn=<ViewBackward0>)

In [46]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d3_01_FT, dem_d3_01_FT, criterion, 2)

  restart 1/5 | best BCE loss so far: 9.0036
  restart 2/5 | best BCE loss so far: 9.0036
  restart 3/5 | best BCE loss so far: 10.0240
  restart 4/5 | best BCE loss so far: 10.0240
  restart 5/5 | best BCE loss so far: 13.1286


In [ ]:
print(best_label)
model_d3_01_FT(best_S)

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d5_005_FT, dem_d5_005_FT, criterion, 2)

  restart 1/5 | best BCE loss so far: 7.7527


In [ ]:
print(best_label)
model_d5_005_FT(best_S)

In [ ]:
best_E, best_loss, best_S, best_label = simulated_annealing.attack(model_d5_01_FT, dem_d5_01_FT, criterion, 2)

In [ ]:
print(best_label)
model_d5_01_FT(best_S)